# Assignment 3 — Parallel & SIMD dense matrix multiplication

Analysis of the stable-session campaign (`seqpre`/`seqpost`-bracketed):

* **Speedup** $S(p) = T_{seq}(n)/T_p(n)$ with $T_{seq}$ the pooled median of the 10 bracketed sequential measurements, $T_p$ the median of 5 batched repetitions.
* **Efficiency** $E(p) = S(p)/p$.
* SIMD comparison from `simd2` (scalar / auto / pragma in one session).
* P-core confinement from `pcore2` vs `scale2`.

In [1]:
from pathlib import Path
import csv
import statistics as st
from collections import defaultdict

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data" / "results").exists():
    ROOT = ROOT.parent
RES = ROOT / "data" / "results"
FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

rows = list(csv.DictReader((RES / "raw_timings.csv").open(encoding="utf-8")))
cfgs = list(csv.DictReader((RES / "configs.csv").open(encoding="utf-8")))

def key(r):
    return (r["experiment"], r["variant"], int(r["n"]),
            int(r["threads"]), r["affinity"])

by = defaultdict(list)
for r in rows:
    by[key(r)].append(float(r["kernel_per_mult_ns"]))

def med(exp, var, n, p, aff):
    return st.median(by[(exp, var, n, p, aff)])

def cv_pct(exp, var, n, p, aff):
    v = by[(exp, var, n, p, aff)]
    return 100.0 * st.pstdev(v) / st.mean(v) if len(v) > 1 and st.mean(v) else 0.0

SIZES = [50, 100, 200, 500, 1000, 2000]
THREADS = [1, 2, 4, 8, 12, 16]
print(f"measurements={len(rows)}  configs={len(cfgs)}  "
      f"all_ok={all(c['status'] == 'ok' for c in cfgs)}")

measurements=1110  configs=222  all_ok=True


## Validation

Same checksum for a given `n` across every variant, thread count and affinity → identical bitwise results (the per-element summation order is unchanged by row partitioning, so this is a strong consistency check across the whole campaign).

In [2]:
chk = defaultdict(set)
for r in rows:
    chk[int(r["n"])].add(r["checksum"])
for n in SIZES:
    print(f"n={n:5d}  distinct checksums across all configs: {len(chk[n])}")
assert all(len(chk[n]) == 1 for n in SIZES), "checksum divergence!"
reps = {len(v) for k, v in by.items() if k[0] in ("scale2", "simd2", "pcore2", "seqpre", "seqpost")}
print("repetitions per stable-session config:", sorted(reps))

n=   50  distinct checksums across all configs: 1
n=  100  distinct checksums across all configs: 1
n=  200  distinct checksums across all configs: 1
n=  500  distinct checksums across all configs: 1
n= 1000  distinct checksums across all configs: 1
n= 2000  distinct checksums across all configs: 1
repetitions per stable-session config: [5]


## Main scaling table — S(p) and E(p) (`scale2`)

In [3]:
def tseq(n):
    vals = by[("seqpre", "seq", n, 1, "all")] + by[("seqpost", "seq", n, 1, "all")]
    return st.median(vals)

scale_rows = []
for n in SIZES:
    ts = tseq(n)
    for p in THREADS:
        tp = med("scale2", "simd", n, p, "all")
        scale_rows.append(dict(n=n, p=p, t_seq_ns=ts, t_p_ns=tp,
                               speedup=ts / tp, efficiency=ts / tp / p,
                               cv_pct=cv_pct("scale2", "simd", n, p, "all")))

print(f"{'n':>5} {'p':>3} {'T_seq(ms)':>10} {'T_p(ms)':>10} {'S(p)':>6} {'E(p)':>6} {'CV%':>5}")
for r in scale_rows:
    print(f"{r['n']:5d} {r['p']:3d} {r['t_seq_ns']/1e6:10.3f} {r['t_p_ns']/1e6:10.3f} "
          f"{r['speedup']:6.2f} {r['efficiency']:6.3f} {r['cv_pct']:5.1f}")

cv = [r["cv_pct"] for r in scale_rows]
print(f"CV across scale2 configs: median={st.median(cv):.1f}%  max={max(cv):.1f}%")
with (RES / "summary_scale.csv").open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(scale_rows[0]))
    w.writeheader(); w.writerows(scale_rows)

    n   p  T_seq(ms)    T_p(ms)   S(p)   E(p)   CV%
   50   1      0.018      0.022   0.79  0.794   4.1
   50   2      0.018      0.045   0.39  0.195  12.7
   50   4      0.018      0.057   0.31  0.077   5.0
   50   8      0.018      0.089   0.20  0.025  10.2
   50  12      0.018      0.097   0.18  0.015   1.5
   50  16      0.018      0.124   0.14  0.009   8.4
  100   1      0.159      0.173   0.92  0.918   4.2
  100   2      0.159      0.160   1.00  0.498   4.1
  100   4      0.159      0.160   1.00  0.249   1.9
  100   8      0.159      0.160   1.00  0.125   1.7
  100  12      0.159      0.216   0.74  0.061   9.1
  100  16      0.159      0.194   0.82  0.051   1.2
  200   1      1.408      1.313   1.07  1.073   8.2
  200   2      1.408      0.917   1.54  0.768   3.9
  200   4      1.408      0.790   1.78  0.446   5.7
  200   8      1.408      0.653   2.16  0.270   3.5
  200  12      1.408      0.660   2.13  0.178   6.9
  200  16      1.408      0.648   2.17  0.136   3.0
  500   1   

In [4]:
fig, ax = plt.subplots(figsize=(6.6, 4.2))
for n in SIZES:
    ts = tseq(n) / 1e6
    tp = [med("scale2", "simd", n, p, "all") / 1e6 for p in THREADS]
    ax.plot(THREADS, tp, marker="o", label=f"n={n} (parallel)")
    ax.axhline(ts, linewidth=0.8, linestyle="--", alpha=0.5)
ax.set_xscale("log", base=2)
ax.set_xticks(THREADS, labels=[str(p) for p in THREADS])
ax.set_yscale("log")
ax.set_xlabel("threads p")
ax.set_ylabel("kernel time per multiplication (ms)")
ax.set_title("Parallel runtime vs thread count\n(dashed: sequential baseline of the same n)")
ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
for ext in ("pdf", "png"):
    fig.savefig(FIG / f"fig1_runtime.{ext}", dpi=150)
plt.close(fig)
print("wrote fig1_runtime.pdf/png")

wrote fig1_runtime.pdf/png


In [5]:
fig, ax = plt.subplots(figsize=(6.6, 4.2))
ideal = list(range(1, 17))
ax.plot(ideal, ideal, linestyle="--", color="black", linewidth=1, label="ideal S(p)=p")
for n in SIZES:
    s = [tseq(n) / med("scale2", "simd", n, p, "all") for p in THREADS]
    ax.plot(THREADS, s, marker="o", label=f"n={n}")
ax.axvline(12, color="tab:red", linewidth=1, linestyle=":")
ax.text(12.1, 0.15, "12 hw contexts", color="tab:red", fontsize=8)
ax.axvline(16, color="tab:red", linewidth=1, linestyle="--", alpha=0.6)
ax.text(15.4, 1.9, "p=16 oversub.", color="tab:red", fontsize=8, ha="right")
ax.set_xscale("log", base=2)
ax.set_xticks(THREADS, labels=[str(p) for p in THREADS])
ax.set_xlabel("threads p")
ax.set_ylabel("speedup S(p) = T_seq / T_p")
ax.set_title("Speedup vs thread count (median of 5 repetitions)")
ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
for ext in ("pdf", "png"):
    fig.savefig(FIG / f"fig2_speedup.{ext}", dpi=150)
plt.close(fig)
print("wrote fig2_speedup.pdf/png")

wrote fig2_speedup.pdf/png


In [6]:
fig, ax = plt.subplots(figsize=(6.6, 4.2))
for n in SIZES:
    e = [tseq(n) / med("scale2", "simd", n, p, "all") / p for p in THREADS]
    ax.plot(THREADS, e, marker="o", label=f"n={n}")
ax.axvline(12, color="tab:red", linewidth=1, linestyle=":", alpha=0.8)
ax.set_xscale("log", base=2)
ax.set_xticks(THREADS, labels=[str(p) for p in THREADS])
ax.set_ylim(0, 1.05)
ax.set_xlabel("threads p")
ax.set_ylabel("efficiency E(p) = S(p)/p")
ax.set_title("Parallel efficiency vs thread count")
ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
for ext in ("pdf", "png"):
    fig.savefig(FIG / f"fig3_efficiency.{ext}", dpi=150)
plt.close(fig)
print("wrote fig3_efficiency.pdf/png")

wrote fig3_efficiency.pdf/png


## Framework overhead at p = 1

In [7]:
print(f"{'n':>5} {'T_seq(us)':>10} {'T_p(1)(us)':>11} {'T_p(1)/T_seq':>13}")
for n in SIZES:
    ts = tseq(n); t1 = med("scale2", "simd", n, 1, "all")
    print(f"{n:5d} {ts/1e3:10.2f} {t1/1e3:11.2f} {t1/ts:13.2f}")

    n  T_seq(us)  T_p(1)(us)  T_p(1)/T_seq
   50      17.52       22.05          1.26
  100     159.27      173.49          1.09
  200    1408.39     1313.17          0.93
  500   21488.46    21981.88          1.02
 1000  233653.80   235686.20          1.01
 2000 3626931.85  3906349.60          1.08


## SIMD study (`simd2`, all three variants in one session)

Thread count held fixed while the vectorisation control varies; evidence in `data/results/vectorisation.txt` (`-fopt-info` + per-symbol objdump counts).

In [8]:
simd_rows = []
for n in [200, 500, 1000, 2000]:
    row = dict(n=n)
    for v in ("scalar", "auto", "simd"):
        for p in (1, 8, 12):
            row[f"{v}_t{p}"] = med("simd2", v, n, p, "all")
    row["gain_p1"] = row["scalar_t1"] / row["simd_t1"]
    row["gain_p8"] = row["scalar_t8"] / row["simd_t8"]
    simd_rows.append(row)

print(f"{'n':>5} {'scalar p1':>10} {'auto p1':>10} {'simd p1':>10} {'S/scalar':>9} {'S/auto':>8} {'gain p8':>8}")
for r in simd_rows:
    print(f"{r['n']:5d} {r['scalar_t1']/1e6:10.2f} {r['auto_t1']/1e6:10.2f} "
          f"{r['simd_t1']/1e6:10.2f} {r['gain_p1']:9.2f} {r['auto_t1']/r['simd_t1']:8.2f} "
          f"{r['gain_p8']:8.2f}")
with (RES / "summary_simd.csv").open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(simd_rows[0]))
    w.writeheader(); w.writerows(simd_rows)

    n  scalar p1    auto p1    simd p1  S/scalar   S/auto  gain p8
  200       3.46       3.47       1.12      3.10     3.11     1.79
  500      60.80      50.79      24.54      2.48     2.07     1.83
 1000     506.99     502.37     254.34      1.99     1.98     1.64
 2000    5080.69    5205.41    3538.06      1.44     1.47     1.47


In [9]:
fig, ax = plt.subplots(figsize=(6.6, 4.2))
ns = [200, 500, 1000, 2000]
for v, mk in (("scalar", "s"), ("auto", "^"), ("simd", "o")):
    ax.plot(ns, [med("simd2", v, n, 1, "all") / 1e6 for n in ns], marker=mk,
            label=f"{v} (p=1)")
ax.plot(ns, [med("seqpre", "seq", n, 1, "all") / 1e6 for n in ns],
        marker="x", linestyle="--", label="sequential baseline (p=1)")
ax.set_xscale("log", base=10); ax.set_yscale("log", base=10)
ax.set_xlabel("n"); ax.set_ylabel("kernel time at p=1 (ms)")
ax.set_title("Vectorisation effect at fixed thread count (p=1)")
ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
for ext in ("pdf", "png"):
    fig.savefig(FIG / f"fig4_simd.{ext}", dpi=150)
plt.close(fig)
print("wrote fig4_simd.pdf/png")

wrote fig4_simd.pdf/png


## P-core confinement (`pcore2`, mask CPU0-3) vs unrestricted (`scale2`)

In [10]:
print(f"{'n':>5} {'p':>3} {'all(ms)':>10} {'pcore(ms)':>10} {'pcore/all':>10}")
pc_rows = []
for n in [500, 1000, 2000]:
    for p in [1, 2, 4]:
        a = med("scale2", "simd", n, p, "all")
        b = med("pcore2", "simd", n, p, "pcore")
        pc_rows.append(dict(n=n, p=p, t_all_ns=a, t_pcore_ns=b, ratio=b / a))
        print(f"{n:5d} {p:3d} {a/1e6:10.2f} {b/1e6:10.2f} {b/a:10.2f}")
with (RES / "summary_pcore.csv").open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(pc_rows[0]))
    w.writeheader(); w.writerows(pc_rows)

    n   p    all(ms)  pcore(ms)  pcore/all
  500   1      21.98      22.05       1.00
  500   2      14.27      15.46       1.08
  500   4      12.70      13.72       1.08
 1000   1     235.69     266.62       1.13
 1000   2     167.64     165.37       0.99
 1000   4     124.11     113.19       0.91
 2000   1    3906.35    3443.15       0.88
 2000   2    2646.38    2280.18       0.86
 2000   4    1916.77    1755.45       0.92


In [11]:
fig, axes = plt.subplots(1, 2, figsize=(9.4, 4.0), sharey=False)
for ax, n in zip(axes, (1000, 2000)):
    ax.plot([1, 2, 4], [med("scale2", "simd", n, p, "all") / 1e6 for p in (1, 2, 4)],
            marker="o", linestyle="-", label="all 12 CPUs")
    ax.plot([1, 2, 3, 4], [med("pcore2", "simd", n, p, "pcore") / 1e6 for p in (1, 2, 3, 4)],
            marker="s", linestyle="--", label="P-cores only (CPU0-3)")
    ax.set_xticks([1, 2, 3, 4])
    ax.set_xlabel("threads p")
    ax.set_title(f"n={n}")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
axes[0].set_ylabel("kernel time (ms)")
fig.suptitle("P-core confinement vs unrestricted scheduling")
fig.tight_layout()
for ext in ("pdf", "png"):
    fig.savefig(FIG / f"fig5_pcore.{ext}", dpi=150)
plt.close(fig)
print("wrote fig5_pcore.pdf/png")

wrote fig5_pcore.pdf/png


## Key numbers for the report

Printed by the cells above:

* `summary_scale.csv` — T_seq, T_p, S(p), E(p), CV% per n and p.
* `summary_simd.csv` — scalar/auto/simd at p in {1, 8, 12} and gains.
* `summary_pcore.csv` — all vs P-cores-only ratios.
* Validation: one checksum per n across every configuration; 5 repetitions per config; CV% distribution shown in the main table cell.